In [11]:
def plot_risk_dashboard(backtest_results):
    """
    Create risk modeling dashboard with 3 panels: VaR comparison, risk vs return, stress test.
    
    Args:
        backtest_results: Dict of {strategy_name: backtest_df}
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('VaR & Risk Modeling Dashboard', fontsize=16, fontweight='bold')
    
    # ===== Plot 1: VaR Comparison =====
    ax = axes[0]
    strategy_names = list(backtest_results.keys())
    var_95_vals = []
    cvar_95_vals = []
    
    for name, backtest_df in backtest_results.items():
        daily_ret = backtest_df['portfolio_value'].pct_change().dropna()
        var_95 = daily_ret.quantile(0.05) * 100
        cvar_95 = daily_ret[daily_ret <= daily_ret.quantile(0.05)].mean() * 100
        var_95_vals.append(var_95)
        cvar_95_vals.append(cvar_95)
    
    x = np.arange(len(strategy_names))
    width = 0.35
    ax.bar(x - width/2, var_95_vals, width, label='VaR (95%)', alpha=0.8, color='orange')
    ax.bar(x + width/2, cvar_95_vals, width, label='CVaR (95%)', alpha=0.8, color='red')
    ax.set_ylabel('Daily Loss (%)')
    ax.set_title('Value-at-Risk Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(strategy_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    # ===== Plot 2: Risk vs Return =====
    ax = axes[1]
    returns = []
    var_95_risks = []
    strategy_colors = ['blue', 'orange', 'green', 'red']
    
    for name, backtest_df in backtest_results.items():
        total_return = backtest_df['portfolio_value'].iloc[-1] / backtest_df['portfolio_value'].iloc[0] - 1
        years = len(backtest_df) / 252
        cagr = (total_return + 1) ** (1/years) - 1
        
        daily_ret = backtest_df['portfolio_value'].pct_change().dropna()
        var_95 = abs(daily_ret.quantile(0.05)) * 100
        
        returns.append(cagr * 100)
        var_95_risks.append(var_95)
    
    for i, name in enumerate(strategy_names):
        ax.scatter(var_95_risks[i], returns[i], s=300, label=name, alpha=0.7, color=strategy_colors[i])
        ax.annotate(name, (var_95_risks[i], returns[i]), xytext=(5, 5), 
                   textcoords='offset points', fontsize=9)
    ax.set_xlabel('Daily Risk (VaR 95%)')
    ax.set_ylabel('Annual Return (CAGR %)')
    ax.set_title('Risk vs Return Trade-off')
    ax.grid(True, alpha=0.3)
    
    # ===== Plot 3: Stress Test Scenarios =====
    ax = axes[2]
    scenarios = {
        '2008 Worst': -0.0916,
        'COVID': -0.1203,
        'Black Monday': -0.2205,
        'Correction': -0.05,
    }
    
    scenario_names_list = list(scenarios.keys())
    max_sharpe_losses = [loss * 100 for loss in scenarios.values()]
    colors_stress = ['darkred' if loss < -0.10 else 'red' if loss < -0.05 else 'orange' 
                     for loss in scenarios.values()]
    
    ax.barh(scenario_names_list, max_sharpe_losses, color=colors_stress, alpha=0.7)
    ax.set_xlabel('Daily Loss (%)')
    ax.set_title('Stress Test: Max Sharpe Portfolio')
    ax.grid(True, alpha=0.3, axis='x')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    
    for i, v in enumerate(max_sharpe_losses):
        ax.text(v - 1, i, f'{v:.2f}%', va='center', ha='right', 
               fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.savefig('risk_dashboard.png', dpi=300, bbox_inches='tight')
    print("✓ Risk dashboard saved to 'risk_dashboard.png'")
    plt.show()